# War Crimes & Human Rights Violations — Exploratory Data Analysis

**Data sources** (all public, no credentials required):
- **UCDP GED 25.1** — Uppsala Georeferenced Event Dataset 1989–2024: state-based conflict, non-state conflict, one-sided violence against civilians (~350K events)
- **HRDAG Colombia** — Documented political killings during the Colombian civil conflict
- **HRDAG Guatemala** — Documented killings during the Guatemalan civil war (1960–1996)
- **UNHCR** — Annual displacement and refugee statistics 2019–2024

Run all cells top-to-bottom. Data is downloaded automatically on first run.

## 0. Install & Import

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', '../requirements.txt', '-q'], check=True)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)
from IPython.display import IFrame, Image
from ingest import ingest_ucdp, ingest_hrdag_colombia, ingest_hrdag_guatemala, ingest_unhcr
from visualize import (
    load_ucdp, OUT_DIR, RAW_DIR, ICC_SITUATION_COUNTRIES, PALETTE,
    chart_choropleth_world, chart_event_cluster_map, chart_heatmap_density,
    chart_monthly_events_by_type, chart_animated_timeseries,
    chart_yoy_violence_civilians, chart_top20_actors, chart_actor_type_by_region,
    chart_actor_network, chart_accountability_gap, chart_data_completeness,
    chart_fatality_analysis, chart_source_analysis, chart_escalation_phases,
    export_summary,
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK. Outputs will be written to:', OUT_DIR)

---
## 1. Ingest Data

Downloads are skipped automatically if the CSVs already exist in `data/raw/`.

In [ ]:
print('Pulling UCDP GED 25.1 (~350 MB on first run)...')
_ = ingest_ucdp()

In [ ]:
print('Pulling HRDAG Colombia...')
_ = ingest_hrdag_colombia()
print('Pulling HRDAG Guatemala...')
_ = ingest_hrdag_guatemala()

In [ ]:
print('Pulling UNHCR displacement...')
_ = ingest_unhcr()

---
## 2. Load & Inspect

In [ ]:
df = load_ucdp(RAW_DIR / 'ucdp_ged.csv')
print(f'UCDP GED: {len(df):,} events | {df["country"].nunique()} countries | {int(df["year"].min())}–{int(df["year"].max())}')
print('\nViolence types:')
print(df['event_type'].value_counts())
print('\nTotal documented fatalities:', f'{df["fatalities"].sum():,}')
df.head(3)

In [ ]:
hrdag_col_path  = RAW_DIR / 'hrdag_colombia.csv'
hrdag_guat_path = RAW_DIR / 'hrdag_guatemala.csv'
df_col  = pd.read_csv(hrdag_col_path,  low_memory=False) if hrdag_col_path.exists()  else pd.DataFrame()
df_guat = pd.read_csv(hrdag_guat_path, low_memory=False) if hrdag_guat_path.exists() else pd.DataFrame()
print(f'HRDAG Colombia:  {len(df_col):,} rows  | columns: {list(df_col.columns[:8])}')
print(f'HRDAG Guatemala: {len(df_guat):,} rows | columns: {list(df_guat.columns[:8])}')

In [ ]:
unhcr_path = RAW_DIR / 'unhcr_displacement.csv'
df_unhcr = pd.read_csv(unhcr_path, low_memory=False) if unhcr_path.exists() else pd.DataFrame()
print(f'UNHCR: {len(df_unhcr):,} rows | columns: {list(df_unhcr.columns)}')

In [ ]:
print('Missing values in UCDP GED:')
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0].to_string())

---
## 3. Global Geographic Distribution

In [ ]:
top20_countries = df['country'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top20_countries.index[::-1], top20_countries.values[::-1], color='#c0392b', alpha=0.85)
ax.set_title('Top 20 Countries by Total Conflict Events (UCDP GED 1989–2024)', fontsize=13)
ax.set_xlabel('Events')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'top20_countries.png', dpi=150)
plt.show()

In [ ]:
chart_choropleth_world(df)
IFrame(str(OUT_DIR / 'choropleth_world.html'), width='100%', height=450)

In [ ]:
chart_event_cluster_map(df)
IFrame(str(OUT_DIR / 'event_cluster_map.html'), width='100%', height=500)

In [ ]:
chart_heatmap_density(df)
IFrame(str(OUT_DIR / 'heatmap_density.html'), width='100%', height=500)

---
## 4. Temporal Patterns

In [ ]:
chart_monthly_events_by_type(df)
IFrame(str(OUT_DIR / 'monthly_events_by_type.html'), width='100%', height=450)

In [ ]:
annual = df.groupby('year').size().rename('events')
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(annual.index.astype(int), annual.values, alpha=0.3, color='#c0392b')
ax.plot(annual.index.astype(int), annual.values, color='#c0392b', linewidth=2)
ax.set_title('Annual Conflict Events (UCDP GED 1989–2024)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Events')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUT_DIR / 'annual_events.png', dpi=150)
plt.show()
print('Peak year:', int(annual.idxmax()), f'({int(annual.max()):,} events)')

In [ ]:
chart_yoy_violence_civilians(df)
Image(str(OUT_DIR / 'yoy_violence_civilians.png'))

In [ ]:
chart_animated_timeseries(df)
IFrame(str(OUT_DIR / 'animated_timeseries.html'), width='100%', height=500)

In [ ]:
chart_escalation_phases(df, top_n_countries=6)
IFrame(str(OUT_DIR / 'escalation_phases.html'), width='100%', height=500)

---
## 5. Actor Analysis

In [ ]:
chart_top20_actors(df)
Image(str(OUT_DIR / 'top20_actors.png'))

In [ ]:
chart_actor_type_by_region(df)
IFrame(str(OUT_DIR / 'actor_type_by_region.html'), width='100%', height=450)

In [ ]:
chart_actor_network(df, top_n=40)
IFrame(str(OUT_DIR / 'actor_network.html'), width='100%', height=600)

---
## 6. Fatality Analysis

In [ ]:
chart_fatality_analysis(df)
Image(str(OUT_DIR / 'fatality_analysis.png'))

In [ ]:
if 'deaths_civilians' in df.columns:
    annual_civ = df.groupby('year')['deaths_civilians'].sum()
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.bar(annual_civ.index.astype(int), annual_civ.values, color='#8e44ad', alpha=0.85)
    ax.set_title('Annual Documented Civilian Deaths (UCDP GED)', fontsize=13)
    ax.set_xlabel('Year')
    ax.set_ylabel('Civilian Deaths')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'civilian_deaths_annual.png', dpi=150)
    plt.show()
    print(f'Total civilian deaths: {int(annual_civ.sum()):,} | Peak year: {int(annual_civ.idxmax())} ({int(annual_civ.max()):,})')

---
## 7. Accountability Gap

In [ ]:
country_counts = df['country'].value_counts().head(30).reset_index()
country_counts.columns = ['country', 'events']
country_counts['icc_situation'] = country_counts['country'].isin(ICC_SITUATION_COUNTRIES)
gap = country_counts[~country_counts['icc_situation']].head(15)
print('Top countries by events with NO open ICC situation:')
print(gap[['country', 'events']].to_string(index=False))

In [ ]:
chart_accountability_gap(df)
IFrame(str(OUT_DIR / 'accountability_gap.html'), width='100%', height=500)

In [ ]:
chart_data_completeness(df)
Image(str(OUT_DIR / 'data_completeness.png'))

---
## 8. Source Transparency

In [ ]:
chart_source_analysis(df)
Image(str(OUT_DIR / 'source_analysis.png'))

In [ ]:
IFrame(str(OUT_DIR / 'source_diversity.html'), width='100%', height=450)

---
## 9. HRDAG Deep Dive

HRDAG uses multiple-systems estimation (MSE) to statistically adjust documented killings for undercount.

In [ ]:
for label, df_h in [('Colombia', df_col), ('Guatemala', df_guat)]:
    if df_h.empty:
        print(f'{label}: no data loaded.')
        continue
    print(f'\n=== HRDAG {label} ===')
    print(f'  Rows: {len(df_h):,}  |  Columns: {df_h.shape[1]}')
    print(f'  Column names: {list(df_h.columns)}')
    print(df_h.describe(include='all').T[['count', 'unique', 'top', 'freq', 'mean']].dropna(how='all').head(12))

In [ ]:
if not df_col.empty:
    year_col = next((c for c in df_col.columns if 'year' in c.lower() or 'año' in c.lower()), None)
    if year_col:
        df_col[year_col] = pd.to_numeric(df_col[year_col], errors='coerce')
        annual_col = df_col.dropna(subset=[year_col]).groupby(year_col).size()
        annual_col = annual_col[annual_col.index.between(1958, 2016)]
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.fill_between(annual_col.index.astype(int), annual_col.values, alpha=0.4, color='#c0392b')
        ax.plot(annual_col.index.astype(int), annual_col.values, color='#c0392b', linewidth=1.8)
        ax.set_title('HRDAG Colombia: Documented Political Killings by Year', fontsize=13)
        ax.set_xlabel('Year')
        ax.set_ylabel('Documented Deaths')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'hrdag_colombia_timeline.png', dpi=150)
        plt.show()
    else:
        print('No year column detected. Available columns:', list(df_col.columns))

---
## 10. UNHCR Displacement

In [ ]:
if df_unhcr.empty:
    print('UNHCR data not available.')
else:
    print('Columns:', list(df_unhcr.columns))
    print(df_unhcr.head())

In [ ]:
if not df_unhcr.empty and 'year' in df_unhcr.columns:
    ref_col = next((c for c in df_unhcr.columns if 'refugee' in c.lower()), None)
    idp_col = next((c for c in df_unhcr.columns if 'idp' in c.lower()), None)
    if ref_col and idp_col:
        annual_disp = df_unhcr.groupby('year')[[ref_col, idp_col]].sum().reset_index()
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(annual_disp['year'], annual_disp[ref_col] / 1e6, label='Refugees', color='#2471a3', alpha=0.8)
        ax.bar(annual_disp['year'], annual_disp[idp_col] / 1e6, bottom=annual_disp[ref_col] / 1e6, label='IDPs', color='#e67e22', alpha=0.8)
        ax.set_title('UNHCR: Global Displacement (Refugees + IDPs) 2019–2024', fontsize=13)
        ax.set_xlabel('Year')
        ax.set_ylabel('People (millions)')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / 'unhcr_displacement.png', dpi=150)
        plt.show()

---
## 11. Summary Export

In [ ]:
print('=== UCDP GED Summary ===')
print(f'  Total events:         {len(df):,}')
print(f'  Countries:            {df["country"].nunique()}')
print(f'  Years:                {int(df["year"].min())}–{int(df["year"].max())}')
print(f'  Total fatalities:     {int(df["fatalities"].sum()):,}')
if 'deaths_civilians' in df.columns:
    print(f'  Civilian deaths:      {int(df["deaths_civilians"].sum()):,}')
print(f'  Unique actors (A):    {df["actor1"].nunique()}')
print()
print(f'HRDAG Colombia:  {len(df_col):,} rows')
print(f'HRDAG Guatemala: {len(df_guat):,} rows')
print(f'UNHCR:           {len(df_unhcr):,} rows')

In [ ]:
export_summary(df)

---
**All outputs saved to `/outputs/`** — interactive HTML charts open directly in a browser.